In [1]:
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM
from transformers import AutoTokenizer
from transformers import GenerationConfig
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import torch
print(torch.version.cuda)

c:\Users\User\anaconda3\envs\summarization\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


12.1


In [2]:
train = pd.read_csv('data/train.csv')

In [3]:
validation = train.iloc[700:]
train = train[:700]

In [4]:
validation.head()

,paper_id,text,summary
700,700,## Introduction\n\n\nCommon sense understandin...,Conventional interpretations of public and soc...
701,701,INTRODUCTION\n\n\nAmerican anthropology is eng...,American Anthropology is engaged in significan...
702,702,## Introduction\n\n\nSwear and taboo words in ...,The current study attempted to determine the t...
703,703,## Introduction\n\n\nAddressing attitudes is c...,Introduction\nAddressing attitudes is central ...
704,704,## Introduction\n\n\nThe concept of time is at...,It is well recognized that time-averaging of a...


In [5]:
validation.shape

(300, 3)

In [6]:
model_name = 'google/flan-t5-large'
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights: 100%|██████████| 558/558 [00:00<00:00, 1362.97it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [7]:
def preprocessing_function(text):
     # quitar referencias tipo [1]
    text = re.sub(r"\[\d+\]", "", text)
    # limpiar caracteres
    text = re.sub(r"[^a-zA-Z0-9\s.,%\-]", " ", text)
    # normalizar espacios
    text = re.sub(r"\s+", " ", text)

    #cluster_chunker = ClusterSemanticChunker(
    #embedding_function=embedding_function,
    #max_chunk_size=400
    # )
      

In [8]:
def cleaning_text(text):
     # quitar referencias tipo [1]
    text = re.sub(r"\[\d+\]", "", text)
    # limpiar caracteres
    text = re.sub(r"[^a-zA-Z0-9\s.,%\-]", " ", text)
    # normalizar espacios
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [9]:
validation['clean_text'] = validation['text'].apply(cleaning_text)

In [10]:
def simple_split(text, chunk_size=60):
    txt_split = text.split()
    return [' '.join(txt_split[i:i+chunk_size]) for i in range(0, len(txt_split), chunk_size)]

In [11]:
sentence_transf_model = SentenceTransformer("BAAI/bge-base-en", device='cuda')
def embedding_function(texts):
    return sentence_transf_model.encode(texts, batch_size=64, show_progress_bar=True)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1783.01it/s]


In [12]:
all_chunks = []
chunk_pos = []

for i, text in enumerate(validation['clean_text']):
    chunks = simple_split(text)
    all_chunks.extend(chunks)
    chunk_pos.extend([i] * len(chunks))


In [13]:
embeddings = sentence_transf_model.encode(
    all_chunks,
    batch_size=64,
    show_progress_bar=True
)

Batches: 100%|██████████| 497/497 [01:42<00:00,  4.83it/s]


In [14]:
def semantic_chunking(chunks, doc_ids, embeddings, window_size=2, t = 0.95, max_len=200):
    #0.9
    merged_chunks = []

    chunks = list(chunks)
    doc_ids = list(doc_ids)
    
    current_chunk = chunks[0]
    current_indices = [0]
    current_doc_id = doc_ids[0]
    
    for i in range(1, len(chunks)):
        
        if doc_ids[i] != current_doc_id:
            merged_chunks.append({
                "text": current_chunk,
                "doc_id": current_doc_id,
                "chunk_indices": current_indices
            })
            
            current_chunk = chunks[i]
            current_indices = [i]
            current_doc_id = doc_ids[i]
            continue
        
        # ventana
        window_indices = current_indices[-window_size:]
        window_embeddings = embeddings[window_indices]
        
        current_embedding = embeddings[i].reshape(1, -1)
        
        sim = cosine_similarity(current_embedding, window_embeddings).max()        
        if sim >= t and len(current_chunk.split()) + len(chunks[i].split()) <= max_len:
            current_chunk += " " + chunks[i]
            current_indices.append(i)
        else:
            merged_chunks.append({
                "text": current_chunk,
                "doc_id": current_doc_id,
                "chunk_indices": current_indices
            })
            
            current_chunk = chunks[i]
            current_indices = [i]
            current_doc_id = doc_ids[i]
    
    # último chunk
    merged_chunks.append({
        "text": current_chunk,
        "doc_id": current_doc_id,
        "chunk_indices": current_indices
    })
    
    return merged_chunks

In [15]:
merged = semantic_chunking(all_chunks, chunk_pos, embeddings)

In [16]:
merged_df = pd.DataFrame(merged)

In [17]:
merged_df_clean = merged_df[merged_df['text'].str.len() > 100]

In [18]:
merged_df_clean.head()

,text,doc_id,chunk_indices
0,Introduction Common sense understandings of pu...,0,[0]
1,literature by using Bacchi s 2009 What s the p...,0,[1]
2,the Council s written documents to further und...,0,[2]
3,"review Foucauldian Discourse Analysis, which f...",0,[3]
4,"of neoliberalism, Thatcherism welfare policy s...",0,[4]


In [19]:
embeddings_chunks = sentence_transf_model.encode(
    merged_df_clean['text'].to_list(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches: 100%|██████████| 493/493 [01:42<00:00,  4.83it/s]


In [20]:
embeddings_chunks.shape

(31548, 768)

In [21]:
summary_embedding = sentence_transf_model.encode(
    validation['summary'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches: 100%|██████████| 5/5 [00:02<00:00,  1.76it/s]


In [22]:
summary_per_chunk = summary_embedding[
    merged_df_clean['doc_id'].values
] # Repetir el summary para cada chunk de text creado

In [23]:
merged_df_clean['scores'] = np.sum(
    embeddings_chunks * summary_per_chunk,
    axis=1
) # Al normalizar los vectores, el cosine_similarity se calcula asi de forma equivalente (mirar formula de cosine similarity)

In [24]:
merged_df_clean.head()

,text,doc_id,chunk_indices,scores
0,Introduction Common sense understandings of pu...,0,[0],0.932996
1,literature by using Bacchi s 2009 What s the p...,0,[1],0.914908
2,the Council s written documents to further und...,0,[2],0.848610
3,"review Foucauldian Discourse Analysis, which f...",0,[3],0.883236
4,"of neoliberalism, Thatcherism welfare policy s...",0,[4],0.812073


In [25]:
merged_df_clean['scores'].mean()

np.float32(0.8491692)

In [26]:
merged_df_clean['scores'].std()

np.float32(0.042923853)

In [27]:
df_sorted = (
    merged_df_clean
    .sort_values(['doc_id','scores'], ascending=[True, False])
    .groupby('doc_id', group_keys=False)
    .head(6)
    #.apply(lambda x: x.head(max(1, int(len(x)*0.2))))
)

In [28]:
df_sorted

,text,doc_id,chunk_indices,scores
0,Introduction Common sense understandings of pu...,0,[0],0.932996
1,literature by using Bacchi s 2009 What s the p...,0,[1],0.914908
45,and social practices and relations that have c...,0,[45],0.896676
61,political actors respond. Discourse as a const...,0,[61],0.891107
3,"review Foucauldian Discourse Analysis, which f...",0,[3],0.883236
...,...,...,...,...
31607,study examined knowledge regarding longleaf pi...,299,[31777],0.943529
31613,significant increase in both understanding and...,299,[31783],0.927201
31611,and acceptance of fire for longleaf pine resto...,299,[31781],0.922572
31624,efforts are currently in place for prescribed ...,299,[31794],0.920998


In [29]:
df_sorted['chunk_indices'] = df_sorted['chunk_indices'].apply(lambda x: min(x)) # Desacemos vector de chunk_indices para ordenar el df por esta variable

In [30]:
df_sorted.sort_values(['chunk_indices'], ascending=True, inplace=True)

In [31]:
df_sorted

,text,doc_id,chunk_indices,scores
0,Introduction Common sense understandings of pu...,0,0,0.932996
1,literature by using Bacchi s 2009 What s the p...,0,1,0.914908
3,"review Foucauldian Discourse Analysis, which f...",0,3,0.883236
42,"to reveal the what could have been, and engage...",0,42,0.876343
45,and social practices and relations that have c...,0,45,0.896676
...,...,...,...,...
31607,study examined knowledge regarding longleaf pi...,299,31777,0.943529
31611,and acceptance of fire for longleaf pine resto...,299,31781,0.922572
31613,significant increase in both understanding and...,299,31783,0.927201
31624,efforts are currently in place for prescribed ...,299,31794,0.920998


In [32]:
df_final = df_sorted.groupby('doc_id')['text'].apply(" ".join)

In [33]:
validation['summary']

700    Conventional interpretations of public and soc...
701    American Anthropology is engaged in significan...
702    The current study attempted to determine the t...
703    Introduction\nAddressing attitudes is central ...
704    It is well recognized that time-averaging of a...
                             ...                        
995    Demand for democratic accountability in polici...
996    Canada’s employment standards laws and mandato...
997    Cultural studies has often favoured a Foucauld...
998    Vehicular air pollution has created an ongoing...
999    For the last several decades, a substantial am...
Name: summary, Length: 300, dtype: str

In [34]:
df_final = pd.DataFrame(df_final)

In [39]:
validation.reset_index(inplace=True)

In [40]:
validation.index

RangeIndex(start=0, stop=300, step=1)

In [41]:
df_final = df_final.join(validation['summary'], how='left')

In [42]:
df_final

,text,summary
doc_id,,
0,Introduction Common sense understandings of pu...,Conventional interpretations of public and soc...
1,is no exception. As researchers and teachers w...,American Anthropology is engaged in significan...
2,Introduction Swear and taboo words in the subt...,The current study attempted to determine the t...
3,Introduction Addressing attitudes is central t...,Introduction\nAddressing attitudes is central ...
4,"Yet, the effects of time-averaging on archaeol...",It is well recognized that time-averaging of a...
...,...,...
295,form a conservative lower-bound for the impact...,Demand for democratic accountability in polici...
296,https www.budget.gc.ca 2016 docs plan budget20...,Canada’s employment standards laws and mandato...
297,Introduction Cultural studies has often attemp...,Cultural studies has often favoured a Foucauld...


In [ ]:
df_final.to_pickle('./model_testing/validation_clean.pkl')

In [44]:
length = df_final['text'].apply(len)

In [45]:
length

doc_id
0      2370
1      2433
2      2438
3      2509
4      2598
       ... 
295    3296
296    2341
297    2433
298    2542
299    3277
Name: text, Length: 300, dtype: int64

In [46]:
length.mean()

np.float64(2533.51)

In [47]:
length.std()

np.float64(342.22281936429295)

In [48]:
np.percentile(merged_df_clean['scores'], [0, 25, 50, 75, 90, 100])

array([0.68323392, 0.82004212, 0.84917283, 0.87822376, 0.90541123,
       0.97999597])